## **Análisis componentes para crear `Namenode`**

### **`Dockerfile`**

**`FROM hadoop-base-image:latest`**: Esta línea indica que la imagen que se va a construir se basará en la imagen base que anteriormente creamos, **hadoop-base-image**. En Docker, una imagen base proporciona el sistema de archivos inicial y los binarios necesarios para ejecutar una aplicación. En este caso, `la imagen base es Ubuntu + Actualizacion de paquetes + Instalacion de paquetes necesarios para Hadoop + Hadoop 3.3.6 + entre otros`.

**`MAINTAINER Alfonso Perez Lino`**: Esta línea se utiliza para especificar el nombre del mantenedor o creador de la imagen. En versiones anteriores de Docker, esta línea era importante para proporcionar información sobre quién mantenía o creaba la imagen. Sin embargo, a partir de Docker 1.13, la instrucción **MAINTAINER** está obsoleta en favor de las etiquetas **LABEL**. Ahora, la información del mantenedor se suele agregar como una etiqueta de metadatos en lugar de utilizar la instrucción **MAINTAINER**.

**`LABEL creador="Alfonso Perez Lino"`**: Esta línea agrega metadatos a la imagen resultante. Los metadatos proporcionan información adicional sobre la imagen, como el nombre del creador, la versión, la descripción, etc. En este caso, se está utilizando la etiqueta creador para indicar el nombre del creador de la imagen. Esta información puede ser útil para otros usuarios que trabajen con la imagen, ya que proporciona contexto sobre su origen y su propósito.

In [ ]:
# Trabajamos sobre la imagen creada anteriormente
FROM hadoop-base-image:latest

LABEL creador="Alfonso Perez Lino"

**`USER root`**: Esta línea indica que se cambiará al usuario root dentro del entorno de la imagen Docker. El usuario root es el superusuario en sistemas basados en Unix y tiene privilegios completos para realizar cualquier acción en el sistema. Al cambiar al usuario root, se obtienen permisos y privilegios completos dentro del entorno de la imagen, lo que permite ejecutar comandos que pueden requerir permisos elevados, como instalar paquetes, configurar el sistema, y realizar otras tareas administrativas.

In [ ]:
# Cambiamos al usuario root
USER root

**`ENV HADOOP_VERSION 3.3.6`**: Esta línea define una variable de entorno llamada **HADOOP_VERSION** con el valor **3.3.6**. Las variables de entorno se utilizan para almacenar información que puede ser utilizada por scripts, aplicaciones o comandos dentro del entorno de la imagen Docker. En este caso, la variable **HADOOP_VERSION** es utilizada para especificar la versión de Hadoop que se utilizará en la imagen.

**`ENV BASE_DIR /opt/bd`**: Esta línea define otra variable de entorno llamada **BASE_DIR** con el valor **/opt/bd**. Esta variable se utiliza para especificar el directorio base en el cual se instalarán o almacenarán componentes relacionados con Big Data en la imagen Docker.

**`ENV LOG_TAG "[NameNode Hadoop_${HADOOP_VERSION}]:"`**: Aquí se define la variable **LOG_TAG** con un valor que parece ser un tag para los logs. Utiliza la variable **HADOOP_VERSION** definida anteriormente para incluir la versión de Hadoop en el tag.

**`ENV HADOOP_HOME ${BASE_DIR}/hadoop/`**: Esta línea establece la variable de entorno **HADOOP_HOME** con el valor **${BASE_DIR}/hadoop/**. **HADOOP_HOME** representa la ubicación del directorio principal de instalación de Hadoop en el contenedor Docker.
Se basa en la variable de entorno **BASE_DIR**, que permite una configuración más flexible y portátil del directorio de instalación de Hadoop.

**`ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/`**: Esta línea establece la variable de entorno **HADOOP_CONF_DIR** con el valor **${HADOOP_HOME}/etc/hadoop/**. **HADOOP_CONF_DIR** representa la ubicación del directorio de configuración de Hadoop en el contenedor Docker. Se basa en la variable de entorno **HADOOP_HOME**, lo que proporciona una forma consistente de definir la ruta del directorio de configuración de Hadoop.

**`ENV DATA_DIR /var/data/hadoop/hdfs`**: Esta línea establece la variable de entorno **DATA_DIR** con el valor **/var/data/hadoop/hdfs**. **DATA_DIR** representa la ubicación del directorio de datos de Hadoop en el contenedor Docker. Este directorio puede ser utilizado para almacenar datos de Hadoop, como datos de HDFS u otros datos relacionados con el funcionamiento de Hadoop.

In [ ]:
# Establecemos las variables
ENV HADOOP_VERSION 3.3.6
ENV BASE_DIR /opt/bd
ENV LOG_TAG "[NameNode Hadoop_${HADOOP_VERSION}]:"
ENV HADOOP_HOME ${BASE_DIR}/hadoop/  # Recordar que este directorio "/opt/bd/hadoop" que corresponde a un "enlace simbolico"
ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/
ENV DATA_DIR /var/data/hadoop/hdfs

**`RUN echo "$LOG_TAG Creando carpetas para los datos de HDFS del NameNode" && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** simplemente imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se están creando carpetas para los datos de HDFS del NameNode. La cadena **$LOG_TAG** corresponde a una variable de entorno que contiene información log adicional, pero no está definida explícitamente en las líneas proporcionadas. Puede ser una variable definida previamente en el Dockerfile o en el entorno de construcción.

**`mkdir -p ${DATA_DIR}/nn`**: Este comando **mkdir** crea un directorio llamado **nn** dentro del directorio especificado por la variable de entorno `${DATA_DIR}`. La opción **-p** indica a **mkdir** que cree los directorios padres si no existen. Esto garantiza que `${DATA_DIR}` y **nn** se creen incluso si `${DATA_DIR}` no existe previamente.

**`chown -R hdadmin:hadoop ${DATA_DIR}`**: Este comando chown cambia el propietario y el grupo del directorio `${DATA_DIR}` y todos sus archivos y subdirectorios al usuario **hdadmin** y al grupo **hadoop**. La opción **-R** indica que la operación se debe realizar de forma recursiva en todos los archivos y subdirectorios dentro de `${DATA_DIR}`. Esto es útil para garantizar que los archivos y directorios necesarios para el funcionamiento de HDFS estén propiedad del usuario y grupo adecuados. 

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log.
- Crean un directorio para los datos del NameNode de HDFS.
- Establecen los permisos adecuados en el directorio de datos de HDFS y sus subdirectorios para que sean propiedad del usuario y grupo correctos (**hdadmin:hadoop**).

In [ ]:
# Creamos las carpetas para los datos de HDFS del NameNode y hacemos el usuario hdadmin propietario de ellas 
# En un entorno de producción se deberían crear varias carpetas en particiones separadas
RUN echo "$LOG_TAG Creando carpetas para los datos de HDFS del NameNode" && \
    mkdir -p ${DATA_DIR}/nn && chown -R hdadmin:hadoop ${DATA_DIR}

**`RUN echo "$LOG_TAG Creando la carpeta para los ficheros de log" && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se está creando un directorio para los archivos log. La cadena **$LOG_TAG** se utilizará como prefijo del mensaje de registro.

**`mkdir ${HADOOP_HOME}/logs`**: Este comando **mkdir** crea un directorio llamado **logs** dentro del directorio especificado por la variable de entorno **${HADOOP_HOME}**. Esto probablemente sea el directorio donde Hadoop almacenará sus archivos log.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log indicando la creación del directorio de archivos log.
- Crean el directorio para los archivos log en la ubicación especificada por **HADOOP_HOME**.

In [ ]:
# Creamos la carpeta para los ficheros de log
RUN echo "$LOG_TAG Creando la carpeta para los ficheros de log" && \
    mkdir ${HADOOP_HOME}/logs

**`RUN echo "$LOG_TAG Copiando ficheros de configuración y script de inicio"`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se están copiando archivos de configuración y script de inicio.

**`COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **core-site.xml** desde la ubicación **Config-files/core-site.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/core-site.xml** en el sistema de archivos del contenedor Docker. **core-site.xml** es un archivo de configuración de Hadoop que contiene configuraciones para la comunicación entre los componentes del clúster Hadoop.

**`COPY Config-files/hdfs-site-namenode.xml ${HADOOP_CONF_DIR}/hdfs-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **hdfs-site-namenode.xml** desde la ubicación **Config-files/hdfs-site-namenode.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/hdfs-site.xml** en el sistema de archivos del contenedor Docker. **hdfs-site-namenode.xml** es otro archivo de configuración de Hadoop que contiene configuraciones específicas del NameNode de Hadoop.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log indicando la copia de archivos de configuración y scripts de inicio.
- Copian archivos de configuración desde el sistema de archivos del host al sistema de archivos del contenedor Docker para configurar adecuadamente el entorno de Hadoop en el contenedor.

**`core.xml`**

El archivo **core-site.xml** es un archivo de configuración importante en el ecosistema de Apache Hadoop. Su función principal es configurar opciones generales que afectan al núcleo del sistema de archivos Hadoop (Hadoop Common). Aquí hay algunas de las configuraciones comunes que pueden encontrarse en **core-site.xml**:

1. `Configuración del sistema de archivos Hadoop (HDFS)`: Esto incluye la ubicación del namenode (el nodo maestro que gestiona el sistema de archivos distribuido) y la configuración de los bloques de datos, como el tamaño predeterminado del bloque y la cantidad de replicas.

2. `Configuración de la seguridad`: Puede incluir configuraciones para la autenticación y autorización, como la elección de mecanismos de autenticación (por ejemplo, Kerberos) y la configuración de la autorización de acceso a los recursos del sistema de archivos.

3. `Configuración de recursos de red y almacenamiento`: Puede incluir configuraciones relacionadas con la utilización de la red, como la configuración del puerto de comunicación, así como configuraciones relacionadas con el almacenamiento, como la ubicación de los directorios temporales.

4. `Configuración del framework de procesamiento de datos MapReduce:` Aunque la mayoría de las configuraciones de MapReduce se realizan en el archivo mapred-site.xml, algunas configuraciones comunes también pueden estar presentes en core-site.xml, como la configuración del framework predeterminado de MapReduce.

5. `Configuración de ubicaciones de logs y directorios temporales`: Puede incluir configuraciones que especifiquen dónde se deben almacenar los registros del sistema y los archivos temporales generados por Hadoop.

En resumen, el archivo core-site.xml es esencial para configurar aspectos fundamentales del sistema de archivos distribuido y otros componentes centrales de Apache Hadoop. Contiene configuraciones críticas que afectan el funcionamiento y el rendimiento del clúster de Hadoop en su conjunto.

**`hdfs-site-datanode.xml`**

El archivo **`hdfs-site.xml`** es un archivo de configuración específico de Apache Hadoop HDFS (Hadoop Distributed File System). Su función principal es definir las configuraciones relacionadas con el sistema de archivos distribuido HDFS en un clúster de Hadoop. Aquí hay una descripción de algunas configuraciones comunes que pueden encontrarse en `hdfs-site.xml` y su función:

1. `Configuración de la replicación de datos`: HDFS almacena los datos en múltiples réplicas para garantizar la tolerancia a fallos y la disponibilidad de datos. En hdfs-site.xml, puedes configurar el número de réplicas que se deben mantener para cada bloque de datos. Esto se realiza mediante la propiedad dfs.replication.

2. `Configuración del tamaño del bloque de datos`: HDFS divide los archivos en bloques de datos de un tamaño fijo y los distribuye en diferentes nodos del clúster. Puedes especificar el tamaño de estos bloques de datos mediante la propiedad dfs.blocksize.

3. `Configuración de la ubicación del almacenamiento de datos`: Puedes especificar la ubicación en el sistema de archivos del sistema de archivos local donde se almacenarán los datos de HDFS. Esto se hace mediante la propiedad dfs.datanode.data.dir.

4. `Configuración de los nombres de los nodos secundarios y de los puntos de montaje`: En un clúster HDFS, puedes tener nodos secundarios que ayudan en la administración del sistema de archivos. Puedes configurar la ubicación de los nombres de los nodos secundarios y los puntos de montaje de HDFS mediante las propiedades dfs.namenode.name.dir y dfs.namenode.edits.dir.

5. `Configuración de cuotas y permisos de acceso`: Puedes configurar cuotas para el espacio de almacenamiento y el número de archivos para los usuarios y grupos, así como establecer permisos de acceso para los archivos y directorios en HDFS.

6. `Configuración de auditoría y registro de eventos`: Puedes habilitar el registro de eventos de auditoría para rastrear las operaciones realizadas en el sistema de archivos HDFS y configurar la ubicación de los registros de auditoría.

En resumen, el archivo hdfs-site.xml es esencial para configurar aspectos fundamentales del sistema de archivos distribuido HDFS en un clúster de Hadoop. Contiene configuraciones críticas que afectan el rendimiento, la disponibilidad y la seguridad de los datos almacenados en HDFS.

In [ ]:
# Copiamos los ficheros de configuración y script de inicio
RUN echo "$LOG_TAG Copiando ficheros de configuración y script de inicio"
COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml
COPY Config-files/hdfs-site-namenode.xml ${HADOOP_CONF_DIR}/hdfs-site.xml

**`COPY Config-files/start-daemons-namenode.sh ${BASE_DIR}/start-daemons.sh`**:Esta línea utiliza la instrucción **COPY** para copiar el archivo **start-daemons-namenode.sh** desde la ubicación **Config-files/start-daemons-namenode.sh** en el sistema de archivos del host al directorio **${BASE_DIR}/start-daemons.sh** en el sistema de archivos del contenedor Docker. **Config-files/start-daemons-namenode.sh** es la ruta en el sistema de archivos del host donde se encuentra el archivo que se desea copiar.

**${BASE_DIR}/start-daemons.sh** es la ruta en el sistema de archivos del contenedor Docker donde se copiará el archivo. **BASE_DIR** es una variable de entorno que se espera que contenga la ruta base del directorio deseado en el contenedor. El archivo se copiará con el nombre **start-daemons.sh** en el directorio especificado.

En resumen, esta línea de comando en el Dockerfile realiza la siguiente acción:

- Copia un archivo desde el sistema de archivos del host al sistema de archivos del contenedor Docker para su uso posterior en el entorno del contenedor.

In [ ]:
# Script de inicio
COPY Config-files/start-daemons-namenode.sh ${BASE_DIR}/start-daemons.sh

**`RUN chmod +x ${BASE_DIR}/start-daemons.sh && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar dos comandos en el contenedor Docker durante la construcción de la imagen. El primer comando, `chmod +x ${BASE_DIR}/start-daemons.sh`, cambia los permisos del archivo **start-daemons.sh** ubicado en el directorio especificado por `${BASE_DIR}` para que sea ejecutable (**+x**). Esto significa que después de ejecutar este comando, el archivo **start-daemons.sh** será ejecutable en el contenedor Docker.
La notación **${BASE_DIR}** se refiere a una variable de entorno que se espera que contenga la ruta base del directorio deseado en el contenedor.

**`chown -R hdadmin:hadoop ${BASE_DIR}`**: El segundo comando establece el propietario y el grupo del directorio `${BASE_DIR}` y todos sus archivos y subdirectorios al usuario **hdadmin** y al grupo **hadoop**. La opción **-R** indica que la operación se realizará de forma recursiva en todos los archivos y subdirectorios dentro de **${BASE_DIR}**. Esto es útil para garantizar que el directorio y sus contenidos sean propiedad del usuario y grupo correctos, lo que puede ser importante para el correcto funcionamiento de ciertas aplicaciones o servicios dentro del contenedor Docker.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Cambian los permisos del archivo **start-daemons.sh** para que sea ejecutable.
- Establecen el propietario y el grupo del directorio **${BASE_DIR}** y sus contenidos al usuario **hdadmin** y al grupo **hadoop**.

In [ ]:
RUN chmod +x ${BASE_DIR}/start-daemons.sh && \
    chown -R hdadmin:hadoop ${BASE_DIR}

**`EXPOSE 8020 9820 9870 9871`**: Esta línea utiliza la instrucción **EXPOSE** para indicar qué puertos deben exponerse desde el contenedor Docker cuando se ejecute una instancia del contenedor. Los números de puerto especificados después de **EXPOSE** (en este caso, 8020, 9820, 9870 y 9871) son los números de puerto que se exponen. En un entorno Hadoop típico, estos puertos son utilizados por diferentes servicios o componentes de Hadoop:

- `8020`: Puerto de FileSystem de Hadoop (HDFS).
- `9820`: Puerto de transferencia de datos secundarios de Hadoop (DataNode).
- `9870`: Puerto de la interfaz de usuario web del NameNode de Hadoop.
- `9871`: Puerto de la interfaz de usuario segura web del NameNode de Hadoop (si está habilitada la seguridad).

Estos puertos están marcados como expuestos, lo que significa que están disponibles para conexiones entrantes desde fuera del contenedor Docker. Sin embargo, exponer un puerto no implica que se mapee automáticamente al sistema host; se necesitará un mapeo explícito de puertos al iniciar el contenedor para redirigir el tráfico del sistema host al contenedor en esos puertos específicos.

In [ ]:
# Preparamos los puertos que usaremos para los servicios web
EXPOSE 8020 9820 9870 9871

La línea de comando `USER hdadmin` en un Dockerfile de Docker se utiliza para establecer el usuario bajo el cual se ejecutarán los comandos y procesos dentro del contenedor Docker cuando se inicie. Aquí está el desglose:

**`USER hdadmin`**: Esta línea establece el usuario predeterminado que se utilizará para ejecutar los comandos y procesos dentro del contenedor Docker. **hdadmin** es el nombre de usuario que se establece como usuario predeterminado. Cuando esta línea se encuentra en un Dockerfile y se construye la imagen, el usuario predeterminado dentro del contenedor se cambiará a **hdadmin** después de que se inicien los contenedores basados en esa imagen. Esto es útil para especificar qué usuario debe ser utilizado dentro del contenedor Docker para fines de seguridad y gestión de permisos. Cambiar al usuario **hdadmin** después de iniciar el contenedor puede ser beneficioso para restringir los privilegios de los procesos dentro del contenedor, especialmente si no es necesario que se ejecuten como el usuario **root** por razones de seguridad.

Es importante tener en cuenta que el usuario **hdadmin** debe existir dentro del contenedor. De lo contrario, se producirán errores durante la construcción o ejecución del contenedor si se especifica un usuario que no existe.

En resumen, la línea de comando `USER hdadmin` en un Dockerfile establece el usuario predeterminado que se utilizará para ejecutar comandos y procesos dentro del contenedor Docker.

In [ ]:
# Cambiamos al usuario hdadmin
USER hdadmin

**`ENV JAVA_HOME /usr/lib/jvm/java-8-openjdk-amd64/`**: Esta línea establece la variable de entorno **JAVA_HOME** con la ruta al directorio de instalación de Java en el contenedor.

**`ENV PATH ${PATH}:${HADOOP_HOME}/bin/:${HADOOP_HOME}/sbin/`**: Esta línea modifica la variable de entorno **PATH** para incluir los directorios donde residen los binarios de Hadoop (`${HADOOP_HOME}/bin/` y `${HADOOP_HOME}/sbin/`). Esto permite que los comandos de Hadoop estén disponibles en la ruta de búsqueda del sistema.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Cambian al usuario hdadmin para ejecutar comandos que no requieren permisos especiales.
- Establecen variables de entorno para configurar el entorno de desarrollo y definir rutas importantes.


**`¿A que se refieren con PATH?`**

La variable de entorno `PATH` en Linux es una cadena que contiene una lista de directorios separados por dos puntos (`:`). Esta variable es utilizada por el sistema operativo para determinar los directorios en los que debe buscar los ejecutables de los comandos cuando se invocan en la línea de comandos.

**Función de la variable PATH**

`1. Localización de Ejecutables:`Cuando un usuario ejecuta un comando en la línea de comandos sin especificar una ruta completa, el sistema operativo busca el ejecutable correspondiente en los directorios listados en la variable PATH, en el orden en que aparecen.

`2. Facilitar la Ejecución de Comandos:`Sin PATH, los usuarios tendrían que proporcionar la ruta completa de cada ejecutable para ejecutar un comando, lo que sería incómodo y poco práctico.

`3. Modularidad y Flexibilidad:` Los usuarios pueden modificar su PATH para incluir directorios personalizados, lo que les permite añadir nuevos comandos sin necesidad de mover los ejecutables a los directorios estándar del sistema.

**Ejemplo de Uso**

Supongamos que la variable **PATH** tiene el siguiente valor:

In [ ]:
/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games

Cuando se ejecuta un comando, por ejemplo, **ls**, el sistema operativo busca el ejecutable **ls** en cada uno de estos directorios en orden:

1. /usr/local/sbin/ls
2. /usr/local/bin/ls
3. /usr/sbin/ls
4. /usr/bin/ls
5. /sbin/ls
6. /bin/ls
7. /usr/games/ls

El primer ejecutable **ls** que encuentra será el que se ejecutará. Si no encuentra **ls** en ninguno de estos directorios, el sistema devolverá un error indicando que el comando no fue encontrado.

**Modificación del PATH**

Los usuarios pueden modificar la variable **PATH** en su sesión actual de la terminal usando el comando export. Por ejemplo:

In [ ]:
export PATH=$PATH:/opt/myprogram/bin

Este comando añade `/opt/myprogram/bin` al final de la lista de directorios en **PATH**. A partir de este momento, el sistema también buscará ejecutables en `/opt/myprogram/bin` cuando se ejecute un comando.

Para hacer cambios permanentes, los usuarios pueden añadir la modificación de **PATH** en su archivo de configuración de shell, como **~/.bashrc** o **~/.profile**. (Esto es para hacerlo en Linux)

In [ ]:
# Definimos las variables
ENV JAVA_HOME /usr/lib/jvm/java-8-openjdk-amd64/
#ENV HADOOP_HOME ${BASE_DIR}/hadoop/ <------- Esto se repite, el usuario "root" ya creó esta variable de entorno
#ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/  <------- Esto se repite, el usuario "root" ya creó esta variable de entorno
ENV PATH ${PATH}:${HADOOP_HOME}/bin/:${HADOOP_HOME}/sbin/

**`WORKDIR ${HADOOP_HOME}`**: Esta línea establece el directorio de trabajo (working directory) predeterminado dentro del contenedor Docker. ${HADOOP_HOME} es una variable de entorno que se espera que contenga la ruta al directorio principal de instalación de Hadoop dentro del contenedor. Al establecer el directorio de trabajo con **WORKDIR**, cualquier comando posterior ejecutado en el Dockerfile se ejecutará en el contexto de este directorio, a menos que se especifique una ruta completa.

**`RUN echo "$LOG_TAG Iniciando servicios"`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando durante la construcción de la imagen. El comando echo simplemente imprime un mensaje en la salida estándar. En este caso, imprime un mensaje que indica que se están iniciando los servicios. La cadena **$LOG_TAG** parece ser una variable que contiene alguna información log adicional, pero no está definida explícitamente en las líneas proporcionadas. Puede ser una variable definida previamente en el Dockerfile o en el entorno de construcción.

**`CMD ["/opt/bd/start-daemons.sh"]`**: Esta línea especifica el comando predeterminado que se ejecutará cuando se inicie un contenedor basado en esta imagen. El comando se especifica como una matriz JSON, donde el primer elemento es el comando a ejecutar (**/opt/bd/start-daemons.sh**). **/opt/bd/start-daemons.sh** parece ser la ruta al script de inicio de los demonios (**daemons**) de Hadoop dentro del contenedor Docker. Este script probablemente sea utilizado para iniciar los servicios de Hadoop necesarios dentro del contenedor. En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

Establecen el directorio de trabajo predeterminado dentro del contenedor.

- Imprimen un mensaje durante la construcción de la imagen.
- Especifican el comando predeterminado que se ejecutará al iniciar un contenedor basado en esta imagen.

In [ ]:
WORKDIR ${HADOOP_HOME} # /opt/bd/hadoop

RUN echo "$LOG_TAG Iniciando servicios"
CMD ["/opt/bd/start-daemons.sh"]

### **Config-files/`core-site.xml`**

El archivo **core-site.xml** en Hadoop es un archivo de configuración clave que define propiedades fundamentales para la operación del clúster Hadoop, incluyendo la configuración del sistema de archivos distribuido de Hadoop (HDFS). Vamos a desglosar y explicar los componentes del ejemplo proporcionado.

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `core-site.xml`**

**Archivo global**, que al momento de arrancar Hadoop les indicará a todos los nodos que el namenode es "**namenode**".

**1. Configuración de fs.defaultFS**
```xml
                                            <property>
                                              <!-- nombre del Namenode -->
                                              <name>fs.defaultFS</name>
                                              <value>hdfs://namenode:9000/</value>
                                              <final>true</final>
                                            </property>
```
**`<name>fs.defaultFS</name>`**: Esta propiedad define el sistema de archivos predeterminado que utilizará Hadoop. **fs.defaultFS** es el nombre de la propiedad.

**`<value>hdfs://namenode:9000/</value>`**: El valor de esta propiedad es la URI del NameNode en el clúster Hadoop. **hdfs://namenode:9000/** indica que el sistema de archivos predeterminado es **HDFS**, y que el NameNode se encuentra en el host namenode escuchando en el puerto **9000**.

- **Ejemplo de URI en Hadoop**

  En el contexto de Hadoop, un ejemplo de URI es **hdfs://namenode:9000/**. Aquí está el desglose:

  - `hdfs`: El esquema indica que se está utilizando el sistema de archivos distribuido de Hadoop (HDFS).
  - `namenode:9000`: La autoridad que especifica el host y el puerto donde está ejecutándose el NameNode.
  - `/`: La ruta, que en este caso es la raíz del sistema de archivos HDFS.

- **Función de URI en Hadoop**

  En Hadoop, las URIs se utilizan para especificar ubicaciones de recursos de manera uniforme. Por ejemplo:

  - **Configuración de fs.defaultFS**:

    - `fs.defaultFS` es una propiedad en core-site.xml que especifica el sistema de archivos predeterminado para Hadoop.
    - `Valor típico`: hdfs://namenode:9000/. Esto indica que el sistema de archivos HDFS está disponible en el host namenode en el puerto 9000.

**`<final>true</final>`**: Esta etiqueta opcional indica que la propiedad no puede ser sobrescrita por configuraciones posteriores. Es decir, su valor es final y no puede ser modificado.

**2. Configuración de hadoop.tmp.dir**
```xml
                                    <property>
                                      <!-- Almacenamiento temporal (debe tener suficiente espacio) -->
                                      <name>hadoop.tmp.dir</name>
                                      <value>/var/tmp/hadoop-${user.name}</value>
                                      <final>true</final>
                                    </property>
```

**`<name>hadoop.tmp.dir</name>`**: Esta propiedad especifica el directorio temporal que utiliza Hadoop para almacenar datos temporales.
**hadoop.tmp.dir** es el nombre de la propiedad.

**`<value>/var/tmp/hadoop-${user.name}</value>`**: El valor de esta propiedad es la ruta al directorio temporal.
`/var/tmp/hadoop-${user.name}` utiliza una variable **${user.name}**, que se expande al nombre del usuario que está ejecutando los procesos de Hadoop. Esto permite que cada usuario tenga su propio directorio temporal separado.

**`<final>true</final>`**: Igual que en el caso anterior, esta etiqueta indica que el valor de la propiedad es final y no puede ser modificado.

**Propósito de Estas Propiedades**

`fs.defaultFS`: Define el sistema de archivos predeterminado que Hadoop utilizará para almacenar y recuperar datos. Configurar correctamente esta propiedad es crucial para la operación del clúster, ya que todos los componentes de Hadoop dependerán de esta URI para acceder a HDFS.

`hadoop.tmp.dir`: Especifica el directorio utilizado para almacenamiento temporal. Este directorio debe tener suficiente espacio para manejar datos temporales utilizados por los procesos de Hadoop, como archivos de intercambio y datos intermedios generados durante las operaciones de procesamiento de datos.

Estas configuraciones son fundamentales para garantizar que Hadoop pueda localizar y utilizar sus recursos de manera eficiente y que el almacenamiento temporal esté configurado adecuadamente para evitar problemas durante la ejecución de trabajos de procesamiento de datos.

### **Config-files/`hdfs-site-namenode.xml`**

El archivo **hdfs-site-namenode.xml** en Hadoop contiene configuraciones específicas para el NameNode del sistema de archivos distribuido de Hadoop (`HDFS`). Cada propiedad en este archivo configura un aspecto diferente del funcionamiento del NameNode. Vamos a desglosar y explicar los componentes proporcionados.

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `hdfs-site-namenode.xml`**

**1. Configuración de `dfs.replication`**

```xml
                                        <property>
                                          <!-- Block replication factor (default value 3) -->
                                          <name>dfs.replication</name>
                                          <value>3</value>
                                          <final>true</final>
                                        </property>
``` 
**`<name>dfs.replication</name>`**: Esta propiedad define el factor de replicación de bloques en HDFS. **dfs.replication** es el nombre de la propiedad.

**`<value>3</value>`**: El valor de esta propiedad es **3**, lo que significa que cada bloque de datos se replicará en tres nodos diferentes en el clúster. Este es el valor predeterminado para proporcionar redundancia y alta disponibilidad.

**`<final>true</final>`**: Esta etiqueta indica que el valor de la propiedad es final y no puede ser sobrescrito por configuraciones posteriores.

**2. Configuración de `dfs.blocksize`**

```xml
                                          <property>
                                            <!-- Block size (default value 128m) -->
                                            <name>dfs.blocksize</name>
                                            <value>64m</value>
                                            <final>true</final>
                                          </property>
```
**`<name>dfs.blocksize</name>`**: Esta propiedad especifica el tamaño de los bloques en HDFS. **dfs.blocksize** es el nombre de la propiedad.

**`<value>64m</value>`**: El valor de esta propiedad es **64m**, lo que significa que el tamaño de cada bloque es de 64 megabytes. El valor predeterminado suele ser 128 megabytes, pero aquí se ha configurado a 64 megabytes.

**`<final>true</final>`**: Igual que en el caso anterior, esta etiqueta indica que el valor de la propiedad es final y no puede ser modificado.

**3. Configuración de `dfs.namenode.name.dir`**

```xml
                                          <property>
                                            <name>dfs.namenode.name.dir</name>
                                            <value>file:///var/data/hadoop/hdfs/nn</value>
                                            <final>true</final>
                                          </property>
```
**`<name>dfs.namenode.name.dir</name>`**: Esta propiedad define la ubicación en el sistema de archivos local donde el NameNode debe almacenar los metadatos. **dfs.namenode.name.dir** es el nombre de la propiedad.

**`<value>file:///var/data/hadoop/hdfs/nn</value>`**: El valor de esta propiedad es **file:///var/data/hadoop/hdfs/nn**, indicando que los metadatos del NameNode se almacenarán en el directorio **/var/data/hadoop/hdfs/nn** en el sistema de archivos local. Se puede configurar una lista separada por comas de directorios para replicar los metadatos en múltiples ubicaciones, proporcionando redundancia.

**`<final>true</final>`**: Esta etiqueta indica que el valor de la propiedad es final y no puede ser sobrescrito.

**4. Configuración de `dfs.namenode.http-address`**

```xml
                                          <property>
                                            <name>dfs.namenode.http-address</name>
                                            <value>namenode:9870</value>
                                          </property>
```
**`<name>dfs.namenode.http-address</name>`**: Esta propiedad define la dirección y el puerto base donde la interfaz web del NameNode estará escuchando. **dfs.namenode.http-address** es el nombre de la propiedad.

**`<value>namenode:9870</value>`**: El valor de esta propiedad es **namenode:9870**, indicando que la interfaz web del NameNode estará accesible en el host namenode en el puerto 9870.

**Propósito de Estas Propiedades**

**`dfs.replication`**: Define el número de réplicas que HDFS mantendrá para cada bloque de datos. Un mayor número de réplicas aumenta la disponibilidad y tolerancia a fallos, pero también requiere más espacio de almacenamiento.

**`dfs.blocksize`**: Especifica el tamaño de cada bloque en HDFS. Un bloque más grande puede mejorar la eficiencia para manejar archivos grandes, mientras que bloques más pequeños pueden ser beneficiosos para archivos más pequeños.

**`dfs.namenode.name.dir`**: Define dónde se almacenan los metadatos del NameNode en el sistema de archivos local. Utilizar múltiples directorios puede mejorar la redundancia y la tolerancia a fallos.

**`dfs.namenode.http-address`**: Especifica la dirección y el puerto en los que la interfaz web del NameNode estará disponible, permitiendo a los administradores monitorear y gestionar HDFS a través de un navegador web.

Estas configuraciones son esenciales para garantizar que HDFS opere de manera eficiente, segura y confiable.

In [ ]:
<configuration>
  <property>
    <!-- Factor de replicación de bloques (valor por defecto 3) -->
    <name>dfs.replication</name>
    <value>3</value>
    <final>true</final>
  </property>
  <property>
   <!-- Tamaño de bloque (valor por defecto 128m) -->
    <name>dfs.blocksize</name>
    <value>64m</value>
    <final>true</final>
  </property>
  <property>
    <!-- Determina en qué parte del sistema de archivos local el DFS NameNode debe almacenar los metadatos. 
         Si se trata de una lista de directorios delimitada por comas, los metadatos se replican en todos los directorios, 
         por redundancia. En un sistema real debería incluir al menos dos directorios:
         uno local en el disco de NameNode y otro montado remotamente (por ejemplo, utilizando NFS) -->
    <name>dfs.namenode.name.dir</name>
    <value>file:///var/data/hadoop/hdfs/nn</value>
    <final>true</final>
  </property>
  <property>
    <!-- La dirección y el puerto base en los que escuchará la interfaz de usuario web de NameNode -->
    <name>dfs.namenode.http-address</name>
    <value>namenode:9870</value>
    <final>true</final>
  </property>
</configuration>

### **Config-files/`start-daemons-namenode.sh`**

Esta línea es el shebang que indica al sistema operativo que use el intérprete de Bash para ejecutar el script:

In [ ]:
#!/bin/bash

Define la variable **HADOOP_HOME**, que apunta al directorio de instalación de Hadoop:

In [ ]:
HADOOP_HOME=/opt/bd/hadoop/

Define las variables **SERVICE** y **DAEMON**. **SERVICE** apunta al binario del comando hdfs de Hadoop, y **DAEMON** especifica el nombre del demonio a gestionar, en este caso, **namenode**:

In [ ]:
SERVICE=${HADOOP_HOME}/bin/hdfs
DAEMON=namenode

**Desglose de la Línea de Comando**

1. `$HADOOP_HOME/bin/hdfs`:

    - `$HADOOP_HOME` es una variable de entorno que contiene la ruta del directorio de instalación de Hadoop. Por ejemplo, podría ser /**opt/bd/hadoop/**.
    - `bin/hdfs` es el comando **hdfs** dentro del directorio bin de la instalación de Hadoop. Este comando se utiliza para interactuar con el sistema de archivos distribuido de Hadoop (HDFS).

2. `namenode -format`:

    - `namenode` especifica que la acción a realizar es sobre el NameNode de Hadoop.
    - `-format` es una opción del comando **namenode** que formatea el NameNode. Esto significa que prepara el NameNode para su primer uso, configurando el sistema de archivos HDFS desde cero.

3. `-nonInteractive`:

    - Esta opción indica que el formato debe realizarse de manera no interactiva. Es decir, no pedirá confirmación al usuario durante el proceso, lo cual es útil en scripts automatizados.
    
4. `2> /dev/null`:

    - `2>` es una redirección de salida estándar en Unix/Linux que redirige el flujo de errores estándar (stderr).
    - `/dev/null` es un dispositivo especial que descarta cualquier dato que se le envíe. En otras palabras, esta parte del comando asegura que cualquier mensaje de error generado durante la ejecución del comando sea ignorado y no se muestre en la salida.

**Propósito de la Línea de Comando**

Formatea el NameNode de Hadoop de manera no interactiva, asegurando que no se solicite ninguna confirmación del usuario durante el proceso. Cualquier mensaje de error que pueda generarse durante esta operación se redirige a /dev/null y, por lo tanto, se descarta.

**Detalles Adicionales**

- **Formateo del NameNode**:

    - El formateo del NameNode es una operación crítica que debe realizarse antes de que el NameNode pueda empezar a gestionar el sistema de archivos HDFS. Esencialmente, crea la estructura inicial necesaria para que HDFS funcione.
    - Este comando debe ejecutarse solo una vez al configurar un nuevo clúster HDFS. Si se ejecuta en un NameNode que ya contiene datos, los datos existentes pueden perderse (aunque la opción -nonInteractive no reformateará si ya hay datos presentes).

- **Redirección de Errores a `/dev/null`**:

    - Esta práctica es común en scripts de automatización para evitar que errores menores o mensajes no críticos interfieran con el flujo de trabajo. Sin embargo, debe usarse con cuidado porque también puede ocultar errores importantes que necesiten atención.

**Ejemplo de Uso**

En el contexto de un script de configuración y arranque, esta línea asegura que el NameNode esté preparado sin requerir interacción del usuario y sin mostrar errores, lo cual es útil para la automatización en entornos de producción o despliegue continuo.

---

**¿Qué es el NameNode en Hadoop?**

El NameNode es un componente crítico del sistema de archivos distribuidos de Hadoop (HDFS). Es el maestro que gestiona el sistema de archivos HDFS y almacena la metadata (información sobre los archivos y directorios) del sistema de archivos. Sin el NameNode, HDFS no puede funcionar porque los DataNodes, que almacenan los datos reales, dependen de la metadata gestionada por el NameNode para saber qué datos almacenar y cómo acceder a ellos.

**¿Qué significa formatear el NameNode?**

Formatear el NameNode es un proceso que se realiza cuando se configura por primera vez el clúster de Hadoop. Esto implica:

**1. Inicialización del Sistema de Archivos**:

- Se crea una nueva estructura del sistema de archivos HDFS.
- El NameNode establece los datos iniciales necesarios para gestionar HDFS.

**2. Creación de la Metadata Inicial**:

- Se crea un nuevo espacio de nombres para HDFS.
- El NameNode establece el directorio base y otros datos necesarios para empezar a gestionar archivos y directorios.

**Detalles del Proceso de Formateo**

Cuando ejecutas el comando:

`$HADOOP_HOME/bin/hdfs namenode -format -nonInteractive`

Esto es lo que sucede:

1. **Preparación del Espacio de Nombres (Namespace)**:

- El NameNode crea y configura el espacio de nombres inicial para el sistema de archivos HDFS.
- Se generan estructuras de datos internas que el NameNode usará para gestionar el espacio de nombres.
- NameNode es una Máquina que contendrá (Almacenará) el espacio de nombres (namespace). La principal responsabilidad del NameNode es almacenar el HDFS namespace. El namespace es una jerarquía de archivos y directorios. Esto significa cosas como el árbol de directorios, permisos de archivos, y el mapeo de archivos a IDs de bloque.

2. **Generación de un ID de Clúster**:

- Se crea un identificador único para el clúster, que es utilizado para asegurar que todos los nodos del clúster pertenezcan al mismo sistema de archivos distribuido.

3. **Creación de los Directorios de Metadata**:

- Se configuran los directorios donde el NameNode almacenará su metadata. Estos directorios pueden estar especificados en la configuración del NameNode (**dfs.namenode.name.dir**).

**¿Cuándo y por qué formatear el NameNode?**

**Configuración Inicial**:

- El formateo se realiza la primera vez que se configura el NameNode en un nuevo clúster de Hadoop. Esto establece la base sobre la cual se almacenarán y gestionarán los datos en HDFS.

**Evitar Reformatos Accidentales**:

- Una vez formateado y en funcionamiento, formatear nuevamente el NameNode destruirá la metadata existente, lo que puede llevar a la pérdida de acceso a los datos almacenados en HDFS. Por eso, formatear es una operación crítica y debe hacerse con mucho cuidado.

---

**Ejemplo de Formateo en un Script**

En el contexto del script:

`$HADOOP_HOME/bin/hdfs namenode -format -nonInteractive 2> /dev/null`

- `-format`: Indica que el NameNode debe ser formateado.
- `-nonInteractive`: Hace que el formateo se realice sin pedir confirmaciones al usuario, ideal para scripts automatizados.
- `2> /dev/null`: Redirige los errores a /dev/null, evitando que se muestren mensajes de error en la salida estándar.

Este comando garantiza que, al iniciar el NameNode por primera vez, se configure correctamente y esté listo para gestionar el sistema de archivos HDFS sin necesidad de intervención manual.

---

**Resumen**

Formatear el NameNode es el paso inicial crucial para poner en marcha un clúster Hadoop. Establece la estructura fundamental del sistema de archivos distribuido y prepara el NameNode para gestionar la metadata esencial para HDFS. Este proceso se realiza solo una vez durante la configuración inicial y debe manejarse con cuidado para evitar la pérdida de datos.

In [ ]:
$HADOOP_HOME/bin/hdfs namenode -format -nonInteractive 2> /dev/null

---

Inicia el demonio del NameNode. La variable **status** captura el código de retorno del comando (**$?**), que es **0** si el comando se ejecuta correctamente. Si **status** no es **0**, imprime un mensaje de error y termina el script con el código de error correspondiente.

**Desglose Línea por Línea**

**1. Iniciar el Servicio**:

`${SERVICE} --daemon start ${DAEMON}`

- `${SERVICE}`: Esta variable contiene la ruta al binario de hdfs (el comando HDFS de Hadoop), que en este caso es ${HADOOP_HOME}/bin/hdfs.
- `--daemon start ${DAEMON}`: Esta parte del comando indica que se debe iniciar un demonio (daemon) y el demonio a iniciar es el namenode (almacenado en la variable ${DAEMON}).

En conjunto, esta línea inicia el NameNode como un proceso en segundo plano (daemon).

**2. Capturar el Código de Retorno**:

`status=$?`

- `$?`: Esta es una variable especial en bash que contiene el código de retorno del último comando ejecutado.
- `status=$?`: Aquí se asigna el valor del código de retorno del comando anterior (es decir, el intento de iniciar el NameNode) a la variable status.

    **Código de Retorno**:

- Un código de retorno de 0 típicamente indica que el comando se ejecutó con éxito.
- Un código diferente de 0 indica que hubo algún error al ejecutar el comando.

**3. Verificar el Estado y Manejar Errores**:

```
if [ $status -ne 0 ]; then
  echo "No pudo inicializar el servicio ${DAEMON}: $status"
  exit $status
fi
```
-`if [ $status -ne 0 ]; then`: Esta línea verifica si el valor de status es diferente de 0 (**-ne** significa "no es igual" en bash).
    
  - Si status no es 0, significa que el comando para iniciar el NameNode falló.

-`echo "No pudo inicializar el servicio ${DAEMON}: $status"`: Si el comando falló, esta línea imprime un mensaje de error indicando que no se pudo inicializar el servicio namenode y muestra el código de error.

- `exit $status`: Termina la ejecución del script y devuelve el mismo código de error (status) que generó el fallo al intentar iniciar el NameNode.

-`fi`: Finaliza el bloque if.


**Resumen del Comportamiento**

**1. Intenta Iniciar el NameNode**:

- `hdfs --daemon start namenode` se ejecuta para iniciar el NameNode como un demonio.

**2. Captura del Código de Retorno**:

- El código de retorno del intento de iniciar el NameNode se captura en la variable status.

**3. Verificación del Éxito o Fallo**:

- Si `status` es diferente de 0, se imprime un mensaje de error y el script termina con el mismo código de error.
- Si `status` es 0, el script continúa su ejecución, ya que el NameNode se inició correctamente.

In [ ]:
# Iniciamos el demonio del namenode y chequeamos si ha arrancado
${SERVICE} --daemon start ${DAEMON}
status=$?
if [ $status -ne 0 ]; then
  echo "No pudo inicializar el servicio ${DAEMON}: $status"
  exit $status
fi

---

Espera a que el demonio del NameNode esté en funcionamiento. Utiliza **ps aux** para listar los procesos, **grep** para buscar el DAEMON y **grep -q -v grep** para excluir el proceso **grep** en sí mismo. Si no encuentra el proceso, espera **1** segundo antes de volver a intentar.

Esta línea de comando en Bash es un `bucle while` que se ejecuta continuamente hasta que no haya ningún proceso llamado "`namenode`" en ejecución. Aquí está el desglose de la línea:

**1. `while`**: Esto inicia un bucle while en Bash. El bucle continuará ejecutando el código dentro de él mientras la condición que sigue a while sea verdadera.

**2. `! ps aux | grep namenode | grep -q -v grep`**: Esta es la condición del bucle while. Analicemos esto en partes:

- **`ps aux`**: Esto lista todos los procesos actualmente en ejecución en el sistema.

- **`|`**: Esto es un "pipe", que toma la salida del comando a la izquierda y la utiliza como entrada para el comando a la derecha.

- **`grep namenode`**: Esto filtra la salida de **ps aux** para mostrar solo las líneas que contienen la cadena "namenode".

- **`| grep -q -v grep`**: Esto filtra aún más las líneas que contienen "namenode", pero excluye las líneas que contienen la cadena "grep" para evitar coincidencias con el propio comando grep. El flag -q hace que grep no imprima nada en la salida, pero solo establece el código de retorno. Si detallamos más esta linea:

    - **|**: Este es el operador de tubería (pipe) en Bash. Toma la salida del comando a la izquierda y la utiliza como entrada para el comando a la derecha.

    - **grep**: Es un comando que se utiliza para buscar patrones en el texto de entrada.

    - **-q**: Este es un flag de grep que significa "quiet" (silencioso). En lugar de imprimir las líneas que coinciden con el patrón, grep no imprime nada y termina tan pronto como se encuentra una coincidencia. Esto es útil cuando solo nos importa si hay una coincidencia o no, sin necesidad de ver la salida.

    - **-v**: Este es otro flag de grep que significa "invertir la coincidencia". En lugar de imprimir las líneas que coinciden con el patrón, grep imprimirá las líneas que no coinciden con el patrón.

    - **grep**: Este es el segundo comando grep, y se utiliza para filtrar las líneas que no contienen el patrón "grep". En este contexto específico, se utiliza para evitar que el propio comando grep aparezca en los resultados de búsqueda.

    Entonces, en resumen, `| grep -q -v grep` se usa para filtrar la salida del comando anterior (en este caso, `ps aux | grep namenode`) para que no contenga ninguna línea que tenga la palabra "`grep`" en ella, evitando así que el propio comando `grep` aparezca como un resultado de la búsqueda.

- **`!`**: Este es el operador de negación en Bash. Hace que la condición sea verdadera si el comando siguiente produce una salida que indica que no se encontraron procesos de "namenode".

**3. `do`**: Esto marca el inicio del bloque de código que se ejecutará mientras la condición sea verdadera.

**4. `sleep 1`**: Esto hace que el script espere 1 segundo antes de volver a verificar si hay procesos "namenode" en ejecución.

**5. `done`**: Esto marca el final del bloque de código que se ejecuta en el bucle while.

En resumen, esta línea de comando espera hasta que no haya ningún proceso llamado "namenode" en ejecución, y luego continúa con el resto del script.

In [ ]:
# Esperamos a que el demonio esté iniciado
while ! ps aux | grep ${DAEMON} | grep -q -v grep
do 
    sleep 1 
done

Espera 5 segundos antes de proceder a la siguiente operación, para asegurarse de que el NameNode esté completamente inicializado.

In [ ]:
# Esperamos 5 segundos antes de crear los directorios
sleep 5

Crea directorios necesarios en HDFS y establece los permisos apropiados:

- **mkdir -p /user/hdadmin**: Crea el directorio **/user/hdadmin** y cualquier directorio padre necesario.
- **mkdir -p /tmp/hadoop-yarn/staging**: Crea el directorio **/tmp/hadoop-yarn/staging** y cualquier directorio padre necesario.
- **chmod -R 1777 /tmp**: Cambia los permisos del directorio **/tmp** y sus subdirectorios a 1777 (permisos de escritura y ejecución para todos los usuarios, y el bit pegajoso).

In [ ]:
# Inicia directorios en HDFS
$HADOOP_HOME/bin/hdfs dfs -mkdir -p /user/hdadmin &&\
$HADOOP_HOME/bin/hdfs dfs -mkdir -p /tmp/hadoop-yarn/staging &&\
$HADOOP_HOME/bin/hdfs dfs -chmod -R 1777 /tmp

Mantiene el contenedor activo mientras el demonio namenode esté en funcionamiento:

- **while true**: Bucle infinito.
- **sleep 10**: Espera 10 segundos en cada iteración del bucle.
- **if ! ps aux | grep ${DAEMON} | grep -q -v grep**: Verifica si el demonio está en ejecución. Si no lo está (! invierte el resultado), imprime un mensaje de error y termina el script con un código de error **1**.

In [ ]:
# Mientras el demonio esté vivo, el contenedor sigue activo
while true
do 
  sleep 10
  if ! ps aux | grep ${DAEMON} | grep -q -v grep
  then
      echo "El demonio ${DAEMON} ha fallado"
      exit 1
  fi
done

**`Resumen`**

Este script automatiza la configuración y ejecución del NameNode de Hadoop dentro de un entorno de contenedor. Realiza las siguientes acciones:

1. Define las variables de entorno necesarias.
2. Formatea el NameNode (sin interactividad y sin reformatear si ya hay datos).
3. Inicia el demonio del NameNode.
4. Verifica si el demonio se ha iniciado correctamente.
5. Crea directorios esenciales en HDFS y establece sus permisos.
6. Mantiene el contenedor activo y monitorea el demonio del NameNode, terminando el contenedor si el demonio falla.